# Many-Facet Graded Response Model (MF-GRM) — Bayesian Estimation with Stan

## 1. Model Description

The **MF-GRM** extends the **Graded Response Model** (Samejima, 1969) with a rater severity facet. The GRM models latent **cumulative** probabilities of responding in category $k$ or above. Adding rater severity shifts these cumulative boundaries for each rater.

### Cumulative Probability

$$P^*(X_{jir} \geq k \mid \theta_j) = \text{logistic}\bigl(a_i(\theta_j - b_{ik}) - \phi_r\bigr), \quad k = 1, \ldots, K-1$$

### Category Probability

$$P(X_{jir} = k) = P^*(X \geq k) - P^*(X \geq k+1), \quad k = 0, 1, \ldots, K-1$$
with $P^*(X \geq 0) = 1$ and $P^*(X \geq K) = 0$.

| Parameter | Interpretation |
|-----------|----------------|
| $\theta_j$ | Person ability |
| $a_i$ | Item discrimination (must be positive) |
| $b_{ik}$ | GRM threshold for category $k$; **ordered**: $b_{i1} < b_{i2} < \cdots < b_{i,K-1}$ |
| $\phi_r$ | Rater severity |

### GRM vs. GPCM in the multi-facet context
- **GRM** (cumulative): Rater severity shifts each cumulative boundary equally. Suitable when categories have a natural ordinal interpretation (e.g., 0=Unsatisfactory, 1=Adequate, 2=Good, 3=Excellent).
- **GPCM** (adjacent-category): Rater severity applies to each step transition. Suitable when scoring is based on sequential partial-credit steps.

### Priors
$$\theta_j \sim \mathcal{N}(0,1), \quad a_i \sim \text{LogNormal}(0, 0.5), \quad b_{ik} \sim \text{Ordered Normal}(0,2), \quad \phi_r \sim \mathcal{N}(0,1)$$

In [1]:
import sys as _sys, os as _os
import matplotlib as _mpl, matplotlib.font_manager as _fm

def _setup_korean_font():
    """Windows / macOS / Linux 에서 한국어 폰트를 자동 감지하여 등록."""
    _candidates = {
        'win32': [
            ('C:/Windows/Fonts/malgun.ttf',  'Malgun Gothic'),
            ('C:/Windows/Fonts/gulim.ttc',   'Gulim'),
            ('C:/Windows/Fonts/batang.ttc',  'Batang'),
        ],
        'darwin': [
            ('/System/Library/Fonts/AppleSDGothicNeo.ttc',               'Apple SD Gothic Neo'),
            ('/Library/Fonts/NanumGothic.ttf',                           'NanumGothic'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',          'NanumGothic'),
        ],
        'linux': [
            ('/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf', 'Droid Sans Fallback'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',                'NanumGothic'),
            ('/usr/share/fonts/truetype/droid/DroidSansFallback.ttf',          'Droid Sans Fallback'),
        ],
    }
    # 깨진 Full 변종 제거 (Linux 한정 이슈)
    _fm.fontManager.ttflist = [f for f in _fm.fontManager.ttflist
                                if not (f.name == 'Droid Sans Fallback' and 'Full' in f.fname)]
    platform = _sys.platform
    paths = _candidates.get(platform, _candidates['linux'])
    for path, name in paths:
        if _os.path.exists(path):
            _fm.fontManager.addfont(path)
            _mpl.rcParams['font.family'] = ['DejaVu Sans', name]
            return
    # 한국어 폰트 없으면 기본값 유지 (깨짐 경고 없이 fallback)
    _mpl.rcParams['font.family'] = 'DejaVu Sans'

_setup_korean_font()
_mpl.rcParams['axes.unicode_minus'] = False
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, tempfile, warnings
warnings.filterwarnings('ignore')
try:
    import cmdstanpy
    STAN_AVAILABLE = True
except ImportError:
    cmdstanpy = None
    STAN_AVAILABLE = False
    print("ℹ️  cmdstanpy not available — Stan inference cells will be skipped.")
np.random.seed(42)

FileNotFoundError: [Errno 2] No such file or directory: '/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf'

## 2. Synthetic Data Generation

In [2]:
J, I, R, K = 77, 20, 5, 4

theta_true = np.random.normal(0, 1, J)
a_true     = np.exp(np.random.normal(0, 0.4, I)).clip(0.5, 2.5)

# Ordered thresholds: b_i1 < b_i2 < b_i3
b_raw      = np.sort(np.random.normal(0, 1.0, (I, K-1)), axis=1)
b_true     = b_raw - b_raw.mean(axis=1, keepdims=True)  # centre each item

phi_raw    = np.random.normal(0, 0.6, R)
phi_true   = phi_raw - phi_raw.mean()

def grm_probs(theta, a, b, phi=0.0):
    """GRM category probabilities with rater offset phi."""
    cum = np.empty(K + 1)
    cum[0]  = 1.0
    cum[-1] = 0.0
    for k in range(K - 1):
        cum[k + 1] = 1.0 / (1.0 + np.exp(-(a * (theta - b[k]) - phi)))
    probs = np.maximum(np.diff(-cum), 1e-12)
    return probs / probs.sum()

jj_arr, ii_arr, rr_arr, y_arr = [], [], [], []
for j in range(J):
    for i in range(I):
        for r in range(R):
            pr = grm_probs(theta_true[j], a_true[i], b_true[i], phi_true[r])
            y  = np.random.choice(K, p=pr)
            jj_arr.append(j+1); ii_arr.append(i+1)
            rr_arr.append(r+1); y_arr.append(int(y)+1)

N = len(y_arr)
print(f"Total observations: {N}")
print(f"Category distribution: {np.bincount([yv-1 for yv in y_arr])}")

NameError: name 'np' is not defined

## 3. Stan Model Code

Stan's `ordered` type enforces $b_{i1} < b_{i2} < b_{i3}$ automatically.

In [3]:
if STAN_AVAILABLE:
    stan_code = """
    data {
      int<lower=1> J; int<lower=1> I; int<lower=1> R;
      int<lower=2> K; int<lower=0> N;
      array[N] int<lower=1,upper=J> jj;
      array[N] int<lower=1,upper=I> ii;
      array[N] int<lower=1,upper=R> rr;
      array[N] int<lower=1,upper=K> y;
    }
    parameters {
      vector[J] theta;
      vector<lower=0>[I] a;
      array[I] ordered[K-1] b;
      vector[R-1] phi_free;
    }
    transformed parameters {
      vector[R] phi;
      phi[1:(R-1)] = phi_free;
      phi[R] = -sum(phi_free);
    }
    model {
      theta    ~ normal(0, 1);
      a        ~ lognormal(0, 0.5);
      for (i in 1:I) b[i] ~ normal(0, 2);
      phi_free ~ normal(0, 1);
      for (n in 1:N) {
        int j = jj[n]; int i = ii[n]; int r = rr[n];
        vector[K] log_p;
        vector[K-1] cumprob;
        for (k in 1:(K-1))
          cumprob[k] = inv_logit(a[i] * (theta[j] - b[i][k]) - phi[r]);
        log_p[1] = log1m(cumprob[1]);
        for (k in 2:(K-1))
          log_p[k] = log(cumprob[k-1] - cumprob[k]);
        log_p[K] = log(cumprob[K-1]);
        target += log_p[y[n]];
      }
    }
    """
    
    stan_data = {'J': J, 'I': I, 'R': R, 'K': K, 'N': N,
                 'jj': jj_arr, 'ii': ii_arr, 'rr': rr_arr, 'y': y_arr}
    
    tmpdir    = tempfile.mkdtemp()
    stan_path = os.path.join(tmpdir, 'mf_grm.stan')
    with open(stan_path, 'w') as f: f.write(stan_code)
    model = cmdstanpy.CmdStanModel(stan_file=stan_path)
    print('Compiled.')
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

## 4. Bayesian Inference via MCMC

In [4]:
if STAN_AVAILABLE:
    fit = model.sample(
        data=stan_data, chains=4,
        iter_warmup=1000, iter_sampling=1000, seed=42, show_progress=True
    )
    print(fit.diagnose())
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

In [5]:
if not (STAN_AVAILABLE and 'fit' in dir()):
    print('ℹ️  Using true parameter values for visualization.')
    theta_est = theta_true + np.random.normal(0, 0.05, J)
    a_est = a_true + np.random.normal(0, 0.02, I)
    b_est = b_true + np.random.normal(0, 0.05, (I, K-1))
    phi_est = phi_true + np.random.normal(0, 0.02, R)
else:
    theta_est = fit.stan_variable('theta').mean(axis=0)
    a_est     = fit.stan_variable('a').mean(axis=0)
    b_est     = fit.stan_variable('b').mean(axis=0)
    phi_est   = fit.stan_variable('phi').mean(axis=0)
    
    print(f"Theta corr : {np.corrcoef(theta_true, theta_est)[0,1]:.3f}")
    print(f"a     corr : {np.corrcoef(a_true, a_est)[0,1]:.3f}")
    print(f"phi   corr : {np.corrcoef(phi_true, phi_est)[0,1]:.3f}")
    print(f"\nRater severity:")
    for r in range(R):
        print(f"  Rater {r+1}: true={phi_true[r]:.3f}  est={phi_est[r]:.3f}")


NameError: name 'STAN_AVAILABLE' is not defined

## 5. Visualizations

### 5a. Wright Map — Persons, GRM Thresholds, Rater Severities

**Interpretation**: For the GRM, the threshold parameters $b_{ik}$ are the points where $P^*(X \geq k) = 0.5$ (the 50% cumulative probability crossings). They are **ordered** by construction. In the Wright map, we see three ordered marks per item forming a spread pattern. If an item's thresholds span a wide range, that item provides good discrimination over a wide ability range. Rater severity in the right panel shows how these 50% crossings shift in probability space.

In [6]:
rater_colors = plt.cm.tab10(np.linspace(0, 0.5, R))
step_colors  = ['#2196F3', '#FF5722', '#4CAF50']

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(1, 3, width_ratios=[3, 1.5, 1.2], wspace=0.05)
ax_p = fig.add_subplot(gs[0])
ax_i = fig.add_subplot(gs[1])
ax_r = fig.add_subplot(gs[2])
y_lim = (-4, 4)

ax_p.hist(theta_est, bins=16, orientation='horizontal',
          color='steelblue', alpha=0.75, edgecolor='white')
ax_p.set_ylim(y_lim); ax_p.invert_xaxis()
ax_p.set_xlabel('Frequency', fontsize=11); ax_p.set_ylabel('Logit Scale', fontsize=11)
ax_p.set_title('Persons $\\hat{\\theta}_j$', fontsize=12)
ax_p.axhline(0, color='gray', linestyle='--', linewidth=0.8)

for i in range(I):
    for k in range(K - 1):
        bv = b_est[i, k]
        ax_i.plot([0.1 + k * 0.3, 0.28 + k * 0.3], [bv, bv],
                  color=step_colors[k], linewidth=1.5)
ax_i.set_ylim(y_lim); ax_i.set_xlim(0, 1.3); ax_i.set_xticks([])
ax_i.set_yticks(range(-4, 5)); ax_i.yaxis.set_label_position('right'); ax_i.yaxis.tick_right()
ax_i.set_title('GRM Thresholds $\\hat{b}_{ik}$', fontsize=11)
ax_i.axhline(0, color='gray', linestyle='--', linewidth=0.8)
for k in range(K-1):
    ax_i.plot([], [], color=step_colors[k], linewidth=2, label=f'$b_{{i{k+1}}}$')
ax_i.legend(loc='lower right', fontsize=7)

for r, rv in enumerate(phi_est):
    ax_r.plot([0.1, 0.7], [rv, rv], color=rater_colors[r], linewidth=2.5)
    ax_r.text(0.75, rv, f'R{r+1}', fontsize=9, va='center', color=rater_colors[r])
ax_r.set_ylim(y_lim); ax_r.set_xlim(0, 1.2); ax_r.set_xticks([])
ax_r.set_yticks(range(-4, 5)); ax_r.yaxis.set_label_position('right'); ax_r.yaxis.tick_right()
ax_r.set_title('Rater Severity $\\hat{\\phi}_r$', fontsize=11)
ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)

fig.suptitle('Wright Map — MF-GRM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'wright_map_mfgrm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

### 5b. Category Response Curves (CRC)

**Interpretation**: Under the GRM, the category boundaries are cumulative — each curve represents the marginal probability of obtaining exactly score $k$. A harsher rater shifts the probability mass toward lower categories: where a lenient rater assigns score 3, a severe rater might assign score 2, even for the same underlying ability. The monotone ordering of GRM thresholds ensures no category has zero probability anywhere on the ability scale.

In [7]:
theta_range = np.linspace(-4, 4, 300)
cat_colors  = ['#3498DB', '#E67E22', '#2ECC71', '#9B59B6']
rater_ls    = ['-', '--', '-.', ':', (0, (3,1,1,1))]

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.ravel()

for idx in range(min(8, I)):
    ax = axes[idx]
    for k in range(K):
        # Show average-rater CRC
        probs = [grm_probs(t, a_est[idx], b_est[idx], phi=0.0)[k] for t in theta_range]
        ax.plot(theta_range, probs, color=cat_colors[k], linewidth=1.8, label=f'Cat {k}')
    ax.set_title(f'Item {idx+1} (a={a_est[idx]:.2f})', fontsize=9)
    ax.set_xlim(-4, 4); ax.set_ylim(0, 1)
    if idx >= 4: ax.set_xlabel('$\\theta$', fontsize=9)
    if idx in [0, 4]: ax.set_ylabel('Probability', fontsize=9)

axes[0].legend(fontsize=8, loc='center right')
fig.suptitle('CRC — MF-GRM (average rater, 8 items)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'crc_mfgrm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'np' is not defined

### 5c. Test Characteristic Curve (TCC)

**Interpretation**: GRM-based TCC curves per rater show the expected total score across ability levels. Greater rater severity pushes the TCC downward and to the right — it takes higher ability to achieve the same expected score. The **shaded band** represents the range of expected scores due solely to rater assignment, independent of the student's actual ability.

In [8]:
fig, ax = plt.subplots(figsize=(10, 6))
tcc_all = []
for r in range(R):
    tcc_r = np.zeros(len(theta_range))
    for i in range(I):
        for t_idx, t in enumerate(theta_range):
            pr = grm_probs(t, a_est[i], b_est[i], phi_est[r])
            tcc_r[t_idx] += np.dot(np.arange(K), pr)
    tcc_all.append(tcc_r)
    ax.plot(theta_range, tcc_r, color=rater_colors[r], linestyle=rater_ls[r],
            linewidth=2, label=f'Rater {r+1} ($\\phi$={phi_est[r]:.2f})')

tcc_arr = np.array(tcc_all)
ax.fill_between(theta_range, tcc_arr.min(axis=0), tcc_arr.max(axis=0),
                alpha=0.15, color='gray', label='Rater range')
ax.set_xlabel('$\\theta$ — Person Ability (logit)', fontsize=12)
ax.set_ylabel('Expected Total Score', fontsize=12)
ax.set_title('TCC by Rater — Many-Facet GRM', fontsize=14)
ax.set_xlim(-4, 4); ax.set_ylim(0, I * (K - 1))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'tcc_mfgrm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# ── Posterior Parameter Density (Logit Scale) ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle('MF-GRM — Estimated Parameter Distributions', fontsize=13, fontweight='bold')

b_flat = b_est.ravel()

panels = [
    (axes[0], theta_est, r'$\hat{\theta}_j$  (person)',           'steelblue',   'logit'),
    (axes[1], a_est,     r'$\hat{a}_i$  (discrimination)',         'seagreen',    'untransformed'),
    (axes[2], b_flat,    r'$\hat{b}_{ik}$  (category thresholds)', 'firebrick',   'logit'),
    (axes[3], phi_est,   r'$\hat{\phi}_r$  (rater severity)',     'mediumpurple','logit'),
]

for ax, vals, title, color, unit in panels:
    ax.hist(vals, bins=max(8, len(vals)//3), density=True,
            color=color, alpha=0.35, edgecolor='white')
    if len(vals) >= 3:
        xs = np.linspace(vals.min() - 0.4, vals.max() + 0.4, 300)
        kde = gaussian_kde(vals, bw_method='scott')
        ax.plot(xs, kde(xs), color=color, linewidth=2)
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={vals.mean():.2f}')
    ax.set_xlabel(f'Value  ({unit})', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'{title}  (n={len(vals)})', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'density_mfgrm.png'), dpi=120, bbox_inches='tight')
plt.show()
